In [ ]:
from google.colab import drive

drive.mount("/content/drive")

Mounted at /content/drive


In [ ]:
# 1. Install DuckDB, Driver Engine, dan JupySQL
!pip install duckdb duckdb-engine jupysql -q

# 2. Aktifkan mode SQL di Colab
%load_ext sql
%config SqlMagic.autopandas = True
%config SqlMagic.feedback = False
%config SqlMagic.displaycon = False

# 3. Hubungkan ke memori database
%sql duckdb:///:memory:

print("🚀 Mesin SQL siap!")

The sql extension is already loaded. To reload it, use:
  %reload_ext sql
🚀 Mesin SQL siap!


In [ ]:
%%sql
CREATE OR REPLACE TABLE coffee_sales AS
SELECT * FROM '/content/drive/MyDrive/Colab Notebooks/Coffee Shop Sales.csv';

,Success


### Kasus Bisnis: Tim operasional ingin melihat 10 transaksi terbesar khusus untuk produk 'Bakery' yang dibeli lebih dari 2 pcs dalam satu transaksi.

In [ ]:
%%sql
SELECT
    transaction_id,
    transaction_date,
    product_category,
    product_detail,
    transaction_qty,
    unit_price,
    ROUND(transaction_qty * unit_price, 2) AS total_bayar
FROM coffee_sales
WHERE product_category = 'Bakery'
  AND transaction_qty > 2
ORDER BY total_bayar DESC
LIMIT 10;

,transaction_id,transaction_date,product_category,product_detail,transaction_qty,unit_price,total_bayar
0,139444,6/22/23,Bakery,Ginger Scone,3,2.65,7.95
1,140540,6/23/23,Bakery,Ginger Scone,3,2.65,7.95
2,143443,6/25/23,Bakery,Ginger Scone,3,2.65,7.95
3,143947,6/26/23,Bakery,Ginger Scone,3,2.65,7.95
4,144129,6/26/23,Bakery,Ginger Scone,3,2.65,7.95
5,144539,6/26/23,Bakery,Ginger Scone,3,2.65,7.95
6,12676,1/23/23,Bakery,Ginger Scone,3,2.65,7.95
7,13504,1/24/23,Bakery,Ginger Scone,3,2.65,7.95
8,14136,1/25/23,Bakery,Ginger Scone,3,2.65,7.95
9,30367,2/23/23,Bakery,Ginger Scone,3,2.65,7.95


### Kasus Bisnis: Berapa total pemasukan (Revenue) dan total kuantitas barang terjual per kategori produk? Urutkan dari yang paling banyak menyumbang uang!

In [ ]:
%%sql
SELECT
    product_category,
    COUNT(transaction_id) AS total_transaksi,
    SUM(transaction_qty) AS total_item_terjual,
    ROUND(SUM(transaction_qty * unit_price), 2) AS total_revenue
FROM coffee_sales
GROUP BY product_category
ORDER BY total_revenue DESC;

,product_category,total_transaksi,total_item_terjual,total_revenue
0,Coffee,58416,89250.0,269952.45
1,Tea,45449,69737.0,196405.95
2,Bakery,22796,23214.0,82315.64
3,Drinking Chocolate,11468,17457.0,72416.00
4,Coffee beans,1753,1828.0,40085.25
5,Branded,747,776.0,13607.00
6,Loose Tea,1210,1210.0,11213.60
7,Flavours,6790,10511.0,8408.80
8,Packaged Chocolate,487,487.0,4407.64


### Kasus Bisnis: Tampilkan produk spesifik (product_detail) apa saja yang total penjualannya sudah menembus lebih dari $10.000!

In [ ]:
%%sql
SELECT
    product_detail,
    ROUND(SUM(transaction_qty * unit_price), 2) AS total_sales
FROM coffee_sales
GROUP BY product_detail
HAVING total_sales > 10000
ORDER BY total_sales DESC;

,product_detail,total_sales
0,Sustainably Grown Organic Lg,21151.75
1,Dark chocolate Lg,21006.00
2,Latte Rg,19112.25
3,Cappuccino Lg,17641.75
4,Morning Sunrise Chai Lg,17384.00
5,Latte,17257.50
6,Jamaican Coffee River Lg,16481.25
7,Sustainably Grown Organic Rg,16233.75
8,Cappuccino,15997.50
9,Brazilian Lg,15109.50


### Kasus Bisnis: Kategorikan setiap transaksi ke dalam 3 level ukuran pesanan: 'Kecil' (1 item), 'Sedang' (2 item), dan 'Besar' (> 2 item), lalu hitung ada berapa banyak transaksi di tiap level.

In [ ]:
%%sql
SELECT
    CASE
        WHEN transaction_qty = 1 THEN 'Kecil (1 pcs)'
        WHEN transaction_qty = 2 THEN 'Sedang (2 pcs)'
        ELSE 'Besar (> 2 pcs)'
    END AS order_size,
    COUNT(transaction_id) AS total_transaksi
FROM coffee_sales
GROUP BY order_size
ORDER BY total_transaksi DESC;

,order_size,total_transaksi
0,Kecil (1 pcs),87159
1,Sedang (2 pcs),58642
2,Besar (> 2 pcs),3315


### Kasus Bisnis: Bagaimana performa penjualan kita dari bulan ke bulan?

In [ ]:
%%sql
SELECT
    STRFTIME(STRPTIME(transaction_date, '%m/%d/%y'), '%Y-%m') AS bulan,
    COUNT(transaction_id) AS total_transaksi,
    ROUND(SUM(unit_price * transaction_qty), 2) AS total_revenue
FROM coffee_sales
GROUP BY bulan
ORDER BY bulan ASC;

,bulan,total_transaksi,total_revenue
0,2023-01,17314,81677.74
1,2023-02,16359,76145.19
2,2023-03,21229,98834.68
3,2023-04,25335,118941.08
4,2023-05,33527,156727.76
5,2023-06,35352,166485.88


### Kasus Bisnis: Hitung rata-rata belanja per transaksi (Average Order Value / AOV) untuk masing-masing cabang toko!

In [ ]:
%%sql
-- 1. Buat tabel sementara bernama 'ringkasan_toko'
WITH ringkasan_toko AS (
    SELECT
        store_location,
        COUNT(DISTINCT transaction_id) AS total_struk,
        SUM(unit_price * transaction_qty) AS total_omzet
    FROM coffee_sales
    GROUP BY store_location
)
-- 2. Tarik datanya dan hitung AOV-nya
SELECT
    store_location,
    total_struk,
    ROUND(total_omzet, 2) AS total_omzet,
    ROUND(total_omzet / total_struk, 2) AS average_order_value
FROM ringkasan_toko
ORDER BY average_order_value DESC;

,store_location,total_struk,total_omzet,average_order_value
0,Lower Manhattan,47782,230057.25,4.81
1,Hell's Kitchen,50735,236511.17,4.66
2,Astoria,50599,232243.91,4.59


### Studi Kasus: Tampilkan Top 3 Kategori Produk Terlaris di MASING-MASING cabang toko!

In [ ]:
%%sql
WITH ranking_produk AS (
    SELECT
        store_location,
        product_category,
        ROUND(SUM(unit_price * transaction_qty), 2) AS total_revenue,
        DENSE_RANK() OVER(
            PARTITION BY store_location
            ORDER BY SUM(unit_price * transaction_qty) DESC
        ) AS rank_posisi
    FROM coffee_sales
    GROUP BY store_location, product_category
)
SELECT
    store_location,
    product_category,
    total_revenue,
    rank_posisi
FROM ranking_produk
WHERE rank_posisi <= 3;

,store_location,product_category,total_revenue,rank_posisi
0,Hell's Kitchen,Coffee,91222.65,1
1,Hell's Kitchen,Tea,64701.30,2
2,Hell's Kitchen,Bakery,27386.95,3
3,Astoria,Coffee,89744.30,1
4,Astoria,Tea,67839.90,2
5,Astoria,Bakery,26599.75,3
6,Lower Manhattan,Coffee,88985.50,1
7,Lower Manhattan,Tea,63864.75,2
8,Lower Manhattan,Bakery,28328.94,3
